# 03 — Run experiments

Three steps: **configure** → **run** → **verify**.

Prerequisite: notebook 02 → `data/annotations/validation_lengths*.csv`


In [ ]:
from pathlib import Path

import pandas as pd

from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments import load_validation_image_ids
from src.experiments.notebook_helpers import (
    RunExperimentsConfig,
    build_experiment_specs,
    preview_experiment_specs,
    project_config_for_experiments,
    resolve_runs_root,
    run_configured_experiments,
)
from src.utils import setup_logging

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
VALIDATION_IDS = load_validation_image_ids(cfg)
print(f"Storage: {STORAGE} | validation images: {len(VALIDATION_IDS)}")


## 1. Configure


In [ ]:
RUN = True  # False = preview only, no execution

RUN_CFG = RunExperimentsConfig(
    # What to run
    pipelines=["baseline"],       # baseline (recommended) | advanced (grid, experimental)
    methods=["skeleton"],         # bbox | pca | skeleton
    splits=["valid"],             # validation CSV IDs live in images/valid
    validation_set_only=True,
    ground_truth_source="validation_lengths2",  # validation_lengths | validation_lengths2

    # Regression (pick one mode)
    run_regression_calibration=False,  # True = train NEW model + run
    use_regression_model=False,        # True = apply saved .joblib (baseline only)
    regression_model_path=None,        # REPO / "outputs/runs/regression_skeleton_validation_lengths2/regression_model.joblib"

    # Advanced only (off by default)
    use_grid_auto_calibration=False,
    use_depth_estimation=False,
    use_3d_measurement=False,

    # Execution
    overwrite=True,
    visualize=False,
    evaluate=True,
    run_name_template="{pipeline}_{method}_v1",
)

SPECS = build_experiment_specs(RUN_CFG)
preview_experiment_specs(SPECS, RUN_CFG)


## 2. Run


In [ ]:
exp_cfg = project_config_for_experiments(RUN_CFG)
runs_root = resolve_runs_root(RUN_CFG)

if RUN:
    summary = run_configured_experiments(RUN_CFG, SPECS, cfg=exp_cfg)
else:
    print("RUN=False — preview only. Set RUN=True to execute.")
    summary = pd.DataFrame()

if len(summary):
    cols = [c for c in ["run_name", "pipeline", "method", "n_predictions", "mae_mm", "rmse_mm", "run_dir"] if c in summary.columns]
    display(summary[cols])


## 3. Verify


In [ ]:
if not len(summary):
    print("Nothing to verify — run step produced no results.")
elif RUN_CFG.validation_set_only and RUN_CFG.image_ids is None:
    n_gt = len(VALIDATION_IDS)
    for _, row in summary.iterrows():
        pred = pd.read_csv(Path(row["predictions_path"]))
        comp_path = row.get("comparison_path")
        comp = pd.read_csv(comp_path) if pd.notna(comp_path) and Path(comp_path).is_file() else None
        msg = f"{row['run_name']}: predictions={len(pred)}"
        if comp is not None:
            ok = len(pred) == n_gt and len(comp) == n_gt
            print(f"{msg}  evaluated={len(comp)}  {'OK' if ok else 'MISMATCH'}")
            if not ok:
                missing = set(VALIDATION_IDS) - set(pred["image_id"].astype(str))
                if missing:
                    print(f"  missing predictions: {sorted(missing)[:5]}...")
        else:
            print(f"{msg}  (no comparison.csv — check ground truth path)")
else:
    print("Skipped — custom image_ids or no runs.")
